## Conditional GAN (CGAN) with MNIST

This notebook provides an implementation of a conditional GAN using the MNIST dataset [Mirza and Osindero](https://arxiv.org/pdf/1411.1784.pdf).


In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import os

In [ ]:
## PyTorch
import torch
import torch.nn as nn
import torch.utils.data as data
from torch.nn.utils.parametrizations import spectral_norm
from torch.optim import Adam
# Torchvision
import torchvision
from torchvision.datasets import MNIST
from torchvision import transforms
from torchvision.utils import make_grid, save_image
import torch.nn.functional as F
# tqdm
from tqdm.notebook import tqdm, trange

In [ ]:
# Configure environment
# Ensure that all operations are deterministic on GPU (if used) for reproducibility
torch.backends.cudnn.determinstic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

### Load and Prepare MNIST Dataset
MNIST is a built in dataset with torchvision and we load it here and prepare dataloaders for PyTorch.

In [ ]:
transform = transforms.Compose([transforms.Pad((2, 2)), transforms.ToTensor()])
mnist_trainset = MNIST(root='./data', train=True, download=True, transform=transform)

In [ ]:
train_batch_size = 256
train_loader = data.DataLoader(mnist_trainset, batch_size=train_batch_size, 
                               shuffle=True, drop_last=True, pin_memory=True)

In [ ]:
def show_tensor_images(image_tensor, 
                       num_images=64, 
                       size=(1, 32, 32), 
                       nrow=8, 
                       filename=""):
    image_unflat = image_tensor.detach().cpu().view(-1, *size)
    image_grid = make_grid(image_unflat[:num_images], nrow=nrow)
    img = image_grid.permute(1, 2, 0).squeeze()
    plt.axis('off')
    plt.imshow(img)
    if len(filename) > 0:
        plt.savefig(filename, dpi=300, bbox_inches='tight')
    plt.show()

### Build the Generator class
The generator class represents the Generator network. The network has a noise vector as input and outputs images of size (3, 32, 32). The network uses four transpose convolutions, three of which have a stride of 2 to expand the image size back up to 32 x 32. The first three layers use Batch Normalization and ReLU activation. The final layer does not have batch normalization and concludes with a Sigmoid function that matches the image scaling to the range [0, 1].

In [ ]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, label_dim = 10, out_size=32, no_channels=1):
        super(Generator, self).__init__()
        self.gen = nn.Sequential(
            nn.ConvTranspose2d( latent_dim + label_dim, out_size * 4, 4, 1, 0, bias=False),
            nn.BatchNorm2d(out_size * 4),
            nn.ReLU(True),
            nn.ConvTranspose2d(out_size * 4, out_size * 2, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_size * 2),
            nn.ReLU(True),
            nn.ConvTranspose2d( out_size * 2, out_size, 4, 2, 1, bias=False),
            nn.BatchNorm2d(out_size),
            nn.ReLU(True),
            nn.ConvTranspose2d( out_size, no_channels, 4, 2, 1, bias=False),
            nn.Sigmoid(),
        )
        
    def forward(self, noise, labels):
        combined = torch.cat((noise, labels), 1)
        return self.gen(combined)

### Build the Discriminator class

The discriminator class represents the Discriminator network that will be trained to determine if an image is generated or from the real data distribution. This is a standard classification network using three dense layers with Leaky ReLU activation (note the final Sigmoid is in the loss function).


In [ ]:
class Discriminator(nn.Module):
    def __init__(self, label_dim = 10, in_size=32, no_channels=1, neg_slope=0.2):
        super(Discriminator, self).__init__()
        self.disc = nn.Sequential(
            spectral_norm(nn.Conv2d(no_channels + label_dim, in_size, 4, 2, 1, bias=False)),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(in_size, in_size * 2, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(in_size * 2),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(in_size * 2, in_size * 4, 4, 2, 1, bias=False)),
            nn.BatchNorm2d(in_size * 4),
            nn.LeakyReLU(0.2, inplace=True),
            spectral_norm(nn.Conv2d(in_size * 4, 1, 4, 1, 0, bias=False)),
        )
        
    def forward(self, image, labels):
        combined = torch.cat((image, labels), 1)
        return self.disc(combined)

### Train the Model
The loss functions must be defined first and then the training loop can be run.

In [ ]:
def gen_loss(gen, disc, batch_size, latent_dim, one_hot_labels, 
             image_labels, bce_loss, device):
    noise = torch.randn(batch_size, latent_dim, 1, 1, device=device)
    fake_im = gen(noise, one_hot_labels)
    fake_pred = disc(fake_im, image_labels)
    gen_loss = bce_loss(fake_pred, torch.ones_like(fake_pred))
    return gen_loss

In [ ]:
def disc_loss(gen, disc, real_im, batch_size, latent_dim, one_hot_labels, 
              image_labels, bce_loss, device):
    noise = torch.randn(batch_size, latent_dim, 1, 1, device=device)
    fake_im = gen(noise, one_hot_labels).detach()
    fake_pred = disc(fake_im, image_labels)
    fake_loss = bce_loss(fake_pred, torch.zeros_like(fake_pred))
    real_pred = disc(real_im, image_labels)
    real_loss = bce_loss(real_pred, torch.ones_like(real_pred))
    disc_loss = 0.5 * (fake_loss + real_loss)
    return disc_loss

In [ ]:
bce_loss = nn.BCEWithLogitsLoss()
n_epochs = 50
latent_dim = 100
lr = 0.0001
beta_1 = 0.5
beta_2 = 0.9
epoch_show = 5
show_batch = 64
n_critic = 1
n_classes = 10

In [ ]:
gen = Generator(latent_dim=latent_dim).to(device)
gen_opt = Adam(gen.parameters(), lr=lr, betas=(beta_1, beta_2))
disc = Discriminator().to(device) 
disc_opt = Adam(disc.parameters(), lr=lr, betas=(beta_1, beta_2))

In [ ]:
gen_loss_lst = list()
disc_loss_lst = list()
tqdm_epoch = trange(n_epochs)
for epoch in tqdm_epoch:
    avg_gen_loss = 0.0
    avg_disc_loss = 0.0
    num_items = 0
    for real_im, real_lbls in tqdm(train_loader):
        cur_batch_size = len(real_im)
        real_im = real_im.to(device)
        one_hot_labels = F.one_hot(real_lbls.to(device), n_classes)
        image_one_hot_labels = one_hot_labels[:, :, None, None]
        image_one_hot_labels = image_one_hot_labels.repeat(1, 1, 32, 32)
        one_hot_labels = torch.unsqueeze(one_hot_labels, 2)
        one_hot_labels = torch.unsqueeze(one_hot_labels, 2)
        
        for dsteps in range(n_critic):
            # Update discriminator
            disc_opt.zero_grad()
        
            dloss = disc_loss(gen, disc, real_im, cur_batch_size, 
                              latent_dim, one_hot_labels,
                              image_one_hot_labels, 
                              bce_loss, device)
            dloss.backward(retain_graph=True)
            disc_opt.step()

        # Update Generator
        gen_opt.zero_grad()
        gloss = gen_loss(gen, disc, cur_batch_size, 
                         latent_dim,  one_hot_labels,
                         image_one_hot_labels,
                         bce_loss, device)
        gloss.backward(retain_graph=True)
        gen_opt.step()

        avg_gen_loss += gloss.item() * cur_batch_size
        avg_disc_loss += dloss.item() * cur_batch_size
        num_items += cur_batch_size
        
    if epoch % epoch_show == 0:
        noise = torch.randn(show_batch, latent_dim, 1, 1, device=device)
        labels = torch.zeros(show_batch, n_classes, 1, 1, device=device)
        
        j = 0
        for i in range(show_batch):
            labels[i,j,0,0] = 1
            j = (j + 1) % 10
        fake_im = gen(noise, labels)
        show_tensor_images(fake_im, filename='CGANMNISTGrid_' + str(epoch) + '.png')
        show_tensor_images(real_im)
        
    ave_gen_loss = avg_gen_loss / num_items
    ave_disc_loss = avg_disc_loss / num_items
    gen_loss_lst.append(ave_gen_loss)
    disc_loss_lst.append(ave_disc_loss)
    tqdm_epoch.set_description('Ave Gen Loss: {:5f}, \
                               Ave Disc Loss: {:5f}'.format(ave_gen_loss, ave_disc_loss))
    torch.save(gen.state_dict(), 'cganmnist_genckpt.pth')
    torch.save(disc.state_dict(), 'cganmnist_discckpt.pth')

### Plot Losses

In [ ]:
epochs = range(1, n_epochs + 1)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))

from cycler import cycler
monochrome = (cycler('color', ['k']) * cycler('linestyle', ['-', '--', ':', '-.']))

ax.set_prop_cycle(monochrome)
ax.plot(epochs, gen_loss_lst, label='Generator Loss')
ax.plot(epochs, disc_loss_lst, label='Discriminator Loss')
    
ax.set_xlabel('Epochs')
ax.set_ylabel('Loss')
ax.legend(loc='center left', bbox_to_anchor=(1, 0.5))
fig.savefig('CGANMNISTLosses.png', dpi=300, bbox_inches='tight')

In [ ]:
noise = torch.randn(100, latent_dim, 1, 1, device=device)
labels = torch.zeros(100, n_classes, 1, 1, device=device)
for i in range(100):
    labels[i, i % 10, :, :] = 1
samples = gen(noise, labels)
plt.figure(figsize=(12,12))
show_tensor_images(samples, num_images = 100, nrow = 10, filename='CGANMNISTGrid.png')